In [4]:
!pip install ipywidgets

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
import time

# ==============================
# PARAMETERS
# ==============================
noise_level = 0.05
error_history = []

# ==============================
# BB84 SIMULATION (simplified)
# ==============================
def run_bb84(eve_attack):

    n_bits = 200

    alice_bits = np.random.randint(0, 2, n_bits)
    alice_bases = np.random.randint(0, 2, n_bits)
    bob_bases = np.random.randint(0, 2, n_bits)

    bob_results = []

    for i in range(n_bits):
        bit = alice_bits[i]
        basis = alice_bases[i]

        # Eve attack
        if np.random.rand() < eve_attack:
            eve_basis = np.random.randint(0, 2)
            if eve_basis != basis:
                bit = np.random.randint(0, 2)

        # Noise
        if np.random.rand() < noise_level:
            bit = 1 - bit

        # Bob measurement
        if basis == bob_bases[i]:
            bob_results.append(bit)
        else:
            bob_results.append(np.random.randint(0, 2))

    # Sifting
    sift_a = []
    sift_b = []

    for i in range(n_bits):
        if alice_bases[i] == bob_bases[i]:
            sift_a.append(alice_bits[i])
            sift_b.append(bob_results[i])

    sift_a = np.array(sift_a)
    sift_b = np.array(sift_b)

    if len(sift_a) == 0:
        return 0

    error_rate = np.sum(sift_a != sift_b) / len(sift_a)
    return error_rate


# ==============================
# LIVE PLOTTING FUNCTION
# ==============================
def update_plot(eve_attack):

    global error_history

    error_rate = run_bb84(eve_attack)
    error_history.append(error_rate)

    avg_error = np.mean(error_history)
    std_dev = np.std(error_history)

    # Detection logic
    if avg_error > noise_level + 2 * std_dev:
        status = "⚠️ Eavesdropping Detected"
        color = 'red'
    else:
        status = "✅ Channel Safe"
        color = 'green'

    # Plot
    # clear_output(wait=True) # This line is removed as output_widget will handle clearing

    plt.figure(figsize=(10,5))
    plt.plot(error_history, marker='o', label="Error Rate")

    plt.axhline(noise_level, linestyle='--', label="Noise Level")
    plt.axhline(noise_level + 2*std_dev, linestyle=':', label="Detection Threshold")

    plt.title("BB84 Live Error Monitoring")
    plt.xlabel("Time (Rounds)")
    plt.ylabel("Error Rate")
    plt.legend()

    plt.grid()
    plt.show()

    print(f"Eve Attack Rate: {eve_attack:.2f}")
    print(f"Current Error: {error_rate:.4f}")
    print(f"Average Error: {avg_error:.4f}")
    print(f"Std Dev: {std_dev:.4f}")
    print(status)


# ==============================
# UI CONTROLS
# ==============================

slider = widgets.FloatSlider(
    value=0.2,
    min=0,
    max=1,
    step=0.05,
    description='Eve Attack:',
    continuous_update=False
)

button = widgets.Button(description="Run Step")

# Create an Output widget to hold the plot and text
output_widget = widgets.Output()

def on_button_click(b):
    with output_widget:
        clear_output(wait=True) # Clear only the output_widget's content
        update_plot(slider.value)

button.on_click(on_button_click)

# Display the controls and the output widget
display(slider, button, output_widget)

FloatSlider(value=0.2, continuous_update=False, description='Eve Attack:', max=1.0, step=0.05)

Button(description='Run Step', style=ButtonStyle())

Output()